## Clean NIFD data and select latest visits

Created by: Varuna & Sahana

Date: 05/07/2025

In [1]:
import pandas as pd
import json

In [2]:
directory_path = "/projectnb/vkolagrp/skowshik/foundation_adrd/adrd-foundation-model/data/nifd"

In [3]:
with open(f"{directory_path}/NIFD_dictionary.json", 'r') as json_file:
    data_dict = json.load(json_file)

In [4]:
nifd = pd.read_csv('/projectnb/vkolagrp/datasets/NIFD/csv/raw/NIFD_Clinical_Data_20200724_updated.csv')
nifd.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDCHBEH,RSMS_FTDADBEH,RSMS_FTDLYING,RSMS_FTDGOODF,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS
0,1_S_0190,2/14/14,CON,UCSF,1,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2/14/14,1.0,2.0
1,1_S_0190,8/29/14,CON,UCSF,2,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8/29/14,1.0,0.0
2,1_S_0190,7/17/15,CON,UCSF,3,8/11/61,1,20,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1_S_0231,3/14/13,CON,UCSF,2,8/25/40,1,20,1,4/3/13,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3/14/13,0.0,0.0
4,1_S_0231,8/26/15,CON,UCSF,3,8/25/40,1,20,1,4/17/15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
nifd.shape

(1021, 123)

In [6]:
def get_latest_visits(data):
    result = data.sort_values(by=['LONI_ID', 'VISIT_NUMBER'], ascending=[True, False])
    result = result.drop_duplicates(subset='LONI_ID', keep='first').reset_index(drop=True)
    return result

nifd_latest = get_latest_visits(nifd)
nifd_latest['COHORT'] = "NIFD"

In [7]:
nifd_latest[nifd_latest['DX'].isna()]

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDADBEH,RSMS_FTDLYING,RSMS_FTDGOODF,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,COHORT
307,2_S_0011,8/24/11,NaN,MAYO,1,8/27/50,1,12,1,8/24/11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NIFD
311,2_S_0015,3/28/12,NaN,MAYO,1,8/5/42,1,16,1,3/28/12,...,NaN,NaN,NaN,NaN,NaN,NaN,3/28/12,8.0,14.0,NIFD
320,2_S_0024,9/26/12,NaN,MAYO,1,8/14/44,1,14,1,9/26/12,...,NaN,NaN,NaN,NaN,NaN,NaN,9/26/12,-5.0,-5.0,NIFD
321,2_S_0025,9/26/12,NaN,MAYO,1,2/5/46,2,13,1,9/26/12,...,NaN,NaN,NaN,NaN,NaN,NaN,9/26/12,-5.0,-5.0,NIFD
322,2_S_0026,10/2/12,NaN,MAYO,1,5/3/56,1,99,99,10/2/12,...,NaN,NaN,NaN,NaN,NaN,NaN,10/2/12,-5.0,-5.0,NIFD
324,2_S_0028,1/23/13,NaN,MAYO,1,4/28/42,1,20,1,1/23/13,...,NaN,NaN,NaN,NaN,NaN,NaN,1/23/13,-5.0,-5.0,NIFD


In [8]:
gamlss_data = pd.read_csv('/projectnb/vkolagrp/projects/adrd_foundation_model/image_proc/gamlss_data/gamlss_thresholded_wtext.csv')
nifd_gamlss = gamlss_data[gamlss_data['COHORT'] == 'NIFD']

In [9]:
nifd_gamlss['LONI_ID'] = nifd_gamlss['ID'].str.split('NIFD_').str[1]

/scratch/9210280.1.cbm.q/ipykernel_2374415/3179193391.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  nifd_gamlss['LONI_ID'] = nifd_gamlss['ID'].str.split('NIFD_').str[1]


In [10]:
nifd_latest.columns

Index(['LONI_ID', 'CLINICAL_LINKDATE', 'DX', 'SITE', 'VISIT_NUMBER', 'DOB',
       'GENDER', 'EDUCATION', 'RACE', 'CDR_DCDATE',
       ...
       'RSMS_FTDADBEH', 'RSMS_FTDLYING', 'RSMS_FTDGOODF', 'RSMS_FTDREGUL',
       'RSMS_FTDSMSCR', 'RSMS_FTDSPSCR', 'MDEXAM_DCDATE', 'MDEXAM_UHDRS',
       'MDEXAM_UPDRS', 'COHORT'],
      dtype='object', length=124)

In [11]:
nifd_wimaging = pd.merge(nifd_latest, nifd_gamlss[['LONI_ID', 'brain_findings_summary']], on = ['LONI_ID'], how='left')

In [12]:
# for k, v in data_dict.items():
#     try:
#         print(k, nifd_wimaging[k].min(), nifd_wimaging[k].max())
#     except:
#         continue

In [13]:
form = {
    'A1': 'Subject Demographics',
    'B3': "Unified Parkinson's Disease Rating Scale (UPDRS)",
    'B4': 'Clinical Dementia Rating (CDR)',
    'B5': 'Neuropsychiatric Inventory',
    'B6': 'Geriatric Depression Scale (GDS)',
    'B7': 'Functional Assessment Scale (FAS)',
    'C': 'Neuropsychological Battery Summary Scores',
    'D1': 'Clinician Diagnosis',
    'I': 'T1 MRI findings',
}

In [14]:
skip_cols = []
for k, v in data_dict.items():
    if v['FORM ID'] in ['U1', 'B4', 'D1'] or v['Type'].lower() == "datetime":
        skip_cols.append(k)

In [15]:
def create_mapped_json(row):
    result = {}
    # skip_cols = list(nifd.columns[nifd.columns.str.contains('CDR')]) + list(nifd.columns[nifd.columns.str.contains('DATE')]) + ['DX', 'CLINICAL_LINKDATE', 'VISIT_NUMBER', 'SITE', 'DOB'] + list(nifd.columns[nifd.columns.str.contains('PSP')]) + list(nifd.columns[nifd.columns.str.contains('PSP')])
    
    for column, value in row.items():
        if column in skip_cols:
            continue
            
        # skip if nan
        if pd.isna(value) or value is None:
            continue
        # convert to number to check if value's negative
        try:
            if float(value) < 0:
                continue  
        except (ValueError, TypeError):
            pass  # Not a number, continue processing
        
        if column not in data_dict:
            continue
        
        form_desc = form[data_dict[column]['FORM ID']]
        if form_desc not in result:
            result[form_desc] = {}
            
        # convert value to appropriate type if specified in dictionary
        if column in data_dict and 'Type' in data_dict[column]:
            try:
                data_type = data_dict[column]['Type']
                if data_type == 'int' or data_type == 'integer':
                    value = int(float(value))
                elif data_type == 'float' or data_type == 'number':
                    value = float(value)
                elif data_type == 'str' or data_type == 'string':
                    # Keep as string, but we already checked for negative values above
                    value = str(value)
                elif data_type == 'bool' or data_type == 'boolean':
                    value = bool(value)
            except (ValueError, TypeError):
                # If conversion fails, skip this value
                continue
            
        if column in data_dict:
            description = data_dict[column]['Description']
            
            # Check if the column has a values key in the dictionary
            if 'Values' in data_dict[column]:
                # print(column, value)
                if str(value) in data_dict[column]['Values']:
                    mapped_value = data_dict[column]['Values'][str(value)]
                    # print(column, value, mapped_value)
                    if mapped_value:
                        result[form_desc][description] = mapped_value
                    # print(result[form_desc])
                # Skip this field if value doesn't match any in the dictionary
                else:
                    continue
            # For columns without Values key in dictionary, add the value directly
            else:
                result[form_desc][description] = value
                
        # add T1 findings if available:
        if 'brain_findings_summary' in row and pd.notna(row['brain_findings_summary']):
            gamlss_description = ("A generalized additive model for location, scale and shape (GAMLSS) was fitted to FastSurfer derived regional volumes in a sample of healthy controls (n=2407) following methods described in the \"Brain charts for the human lifespan\" paper published in Nature (2022). Briefly, data were pooled from four cohorts, and subjects without neurodegenerative disease were selected as healthy controls. Regions of interests (ROIs) were constructed for the temporal, parietal, frontal, and occipital lobes as well as the ventricles. Quality check ensured removal of subjects where brain volumes were abnormally high or low (3.5 zscore threshold) to account for FastSurfer processing errors. The Brain charts pipeline was run to fit a normative, sex-stratified growth curve for each ROI. Centile scores were computed for all subjects, including healthy controls and patients. Individuals with centile scores between the 10th and 25th percentile were flagged as having mild atrophy; those with centile scores between the 5th-10th percentile as having moderate atrophy; and those with scores below the 5th percentile as having severe atrophy. The direction was flipped for ventricular ROIs, with enlargement signifying abnormality. The findings for this subject were as follows:\n\n")
            result["T1 MRI findings"] = gamlss_description + str(row['brain_findings_summary'])
            
    result = {k: v for k, v in result.items() if v}
    key_order = [k for k in form.values() if k in result]
    result = {k: result[k] for k in key_order if k in result}
                
    return json.dumps(result)

In [16]:
def create_diag_summary(row):
    return json.dumps({data_dict['DX']['Description']: row['DX']})

In [17]:
nifd_wimaging['patient_summary'] = nifd_wimaging.apply(create_mapped_json, axis=1)
nifd_wimaging['diag_summary'] = nifd_wimaging.apply(create_diag_summary, axis=1)

In [18]:
nifd_wimaging.head()

,LONI_ID,CLINICAL_LINKDATE,DX,SITE,VISIT_NUMBER,DOB,GENDER,EDUCATION,RACE,CDR_DCDATE,...,RSMS_FTDREGUL,RSMS_FTDSMSCR,RSMS_FTDSPSCR,MDEXAM_DCDATE,MDEXAM_UHDRS,MDEXAM_UPDRS,COHORT,brain_findings_summary,patient_summary,diag_summary
0,1_S_0001,8/27/14,CON,UCSF,7,12/9/45,1,15,1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NIFD,NaN,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""CON""}"
1,1_S_0002,1/24/12,BV,UCSF,3,12/28/54,2,13,1,1/24/12,...,NaN,NaN,NaN,NaN,NaN,NaN,NIFD,NaN,"{""Subject Demographics"": {""Gender"": ""Female"", ...","{""Diagnosis"": ""BV""}"
2,1_S_0003,8/20/12,SV,UCSF,4,6/8/44,1,12,1,8/15/12,...,NaN,NaN,NaN,8/15/12,12.0,14.0,NIFD,Hippocampus region has severe atrophy; Tempora...,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""SV""}"
3,1_S_0005,8/26/14,BV,UCSF,6,11/22/51,1,18,1,8/26/14,...,2.0,2.0,12.0,8/26/14,-5.0,15.0,NIFD,NaN,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""BV""}"
4,1_S_0006,7/15/11,BV,UCSF,4,7/24/48,1,15,99,NaN,...,NaN,NaN,NaN,7/15/11,-5.0,3.0,NIFD,NaN,"{""Subject Demographics"": {""Gender"": ""Male"", ""Y...","{""Diagnosis"": ""BV""}"


In [19]:
nifd_wimaging['DX'].value_counts()

DX
CON                136
BV                  77
PNFA                40
SV                  39
PATIENT (OTHER)     39
L_SD                 4
Name: count, dtype: int64

In [20]:
json.loads(nifd_wimaging.iloc[2]['patient_summary'])

{'Subject Demographics': {'Gender': 'Male',
  'Years of Education': 12,
  'Race': 'White'},
 "Unified Parkinson's Disease Rating Scale (UPDRS)": {"MD Exam - Unified Huntington's Disease Rating Scale": 12,
  "MD Exam - Unified Parkinson's Disease Rating Scale": 14},
 'Neuropsychiatric Inventory': {'Neuropsychiatric Inventory - Q Form - Delusions': 'No',
  'Neuropsychiatric Inventory - Q Form - Hallucination Distress': 0,
  'Neuropsychiatric Inventory - Q Form - Agitation': 'No',
  'Neuropsychiatric Inventory - Q Form - Depression': 'No',
  'Neuropsychiatric Inventory - Q Form - Anxiety': 'No',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Elation/Euphoria Severity': 2,
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Apathy/Indifference Severity': 3,
  'Neuropsychiatric Inventory - Q Form - Disinhibition': 'Yes',
  'Neuropsychiatric Inventory - Q Form - Disinhibiti

In [21]:
nifd_wimaging.to_csv(f'{directory_path}/NIFD_wjson.csv', index=False)

In [22]:
nifd_wimaging = nifd_wimaging[nifd_wimaging['DX'].isin(['CON', 'BV', 'PNFA', 'SV'])].reset_index(drop=True)

In [23]:
nifd_wimaging.to_csv(f'{directory_path}/train.csv', index=False)